# Zero-shot / Linear probe（`linear`）—— 单方法 notebook

从 `cs604_vit_final.ipynb` 拆出来，只保留**基线方法**用到的代码。断线续跑、时间预算控制、翻转 TTA、写提交文件这些通用功能只在主 notebook 里。

> Phase 2 的测试集预测只能用主 notebook 的 `PHASE2_TEST` 开关（Phase 1 的同一个模型），不要用这里的 notebook。

**原理**：主干冻结，不做梯度训练，所以这个 notebook 没有训练循环。分类头由 `HEAD_INIT` 决定：
- `proto`（zero-shot）：把每一类训练特征的平均方向当作这一类的权重（类原型），再乘一个温度。不训练任何参数。
- `lp`（linear probe）：从类原型出发，在提前算好的特征上把分类头训练 `LP_EPOCHS` 个 epoch。

**怎么跑**：直接全部运行，会连着跑两个实验，先 zero-shot，再 linear probe。第二个复用第一个提好的特征，但提特征的时间照样算进它的 GPU 时间。结果在 Drive 的 `cs604/runs/<批次名>_1_linear_proto/` 和 `<批次名>_2_linear_lp/`。

**我们的结果**：zero-shot holdout **43.95%**（PDF 参考成绩 43.7%）；linear probe holdout **62.07%**。

**代码重点看**：§6 的 `class_prototypes`、`fit_logit_scale`、`linear_probe`、`init_head`。

## 1. Config

`EXPERIMENTS` 里每个 dict 是一次实验，只写和下面默认值不同的设置。这里跑两次：zero-shot 和 linear probe。

In [ ]:
# ===== Experiments to run: each dict is one run and lists only the settings that differ from the defaults below =====
EXPERIMENTS = [
    dict(HEAD_INIT="proto"),  # zero-shot; the earlier run 0928_0636_1_linear got holdout 43.95%
    dict(HEAD_INIT="lp"),     # linear probe, reusing the features of the first run; earlier lp runs got holdout 62.07%
]

# ===== Defaults =====
METHOD = "linear"
MODEL_NAME = "vit_base_patch16_224.augreg2_in21k_ft_in1k"  # timm name of the base model in the doc (IN21K→IN1K)
HOLDOUT_PER_CLASS = 2    # images per class held out from train for local evaluation (the official val set has no labels)
HEAD_INIT = "proto"      # 'proto': class prototypes (zero-shot) | 'lp': linear probe trained from the prototypes
BATCH_SIZE = None        # None = pick based on VRAM (256 on a 40 or 80 GB GPU); the backbone sees 2 × BATCH_SIZE images per batch
LP_EPOCHS = 10           # linear probe: passes over the cached training features
LP_LR = 1e-3             # linear probe: AdamW learning rate, cosine decay to 0 without warmup
LP_WD = 1e-4             # linear probe: AdamW weight decay
LP_BATCH = 4096          # linear probe: features per step (only a 768 → 10,000 linear layer is trained, so a big batch fits)
LABEL_SMOOTHING = 0.1    # linear probe: label smoothing of the cross-entropy loss
SEED = 0
NUM_CLASSES = 10000

# ===== Paths =====
DRIVE_DIR = "/content/drive/MyDrive/cs604"   # results are saved here
DRIVE_DATA_DIR = f"{DRIVE_DIR}/data"         # where the dataset is on Drive (save it there first with cs604_download_to_drive.ipynb)
DATA_DIR = "/content/data"                   # each session extracts the data to the local Colab disk; the backbone reads images from here
DISCONNECT_WHEN_DONE = False  # release the runtime when done to stop using compute units (handy when running in the background)

## 2. Setup

安装库，挂载 Google Drive，检查 GPU（需要支持 bf16 的 A100、L4 或 H100），准备日志。所有输出都存在 Drive 上，断线也不会丢。

In [ ]:
# Colab already has torch / torchvision; install timm here
!pip install -q timm

In [ ]:
# Settings an experiment in EXPERIMENTS may override
RUN_KEYS = ("MODEL_NAME", "HOLDOUT_PER_CLASS", "HEAD_INIT", "BATCH_SIZE", "LP_EPOCHS", "LP_LR", "LP_WD",
            "LP_BATCH", "LABEL_SMOOTHING", "SEED", "NUM_CLASSES")
BASE_CONFIG = {k: globals()[k] for k in RUN_KEYS}

import os, gc, json, math, time, random, shutil, subprocess
from datetime import datetime
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
from tqdm.auto import tqdm
from google.colab import drive

# ---- Mount Google Drive: data is read from here, and results are saved here, so nothing is lost on disconnect ----
drive.mount("/content/drive")
OUT_ROOT = DRIVE_DIR

# ---- GPU: needs bf16 support, i.e. an A100, L4 or H100 runtime ----
assert torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8, (
    "Pick a GPU runtime with bf16 support: A100, L4 or H100")
DEVICE = "cuda"
props = torch.cuda.get_device_properties(0)
GPU_NAME = props.name
gib = props.total_memory / 1024**3
VRAM_GB = next((v for v in (8, 12, 16, 24, 32, 40, 48, 80, 94, 96, 141, 192) if v >= 0.98 * gib), round(gib))  # nominal VRAM
torch.backends.cuda.matmul.allow_tf32 = True  # TF32 for float32 matrix multiplies: much faster, accuracy barely changes
torch.backends.cudnn.allow_tf32 = True
NUM_WORKERS = min(16, os.cpu_count())  # processes that read and decode images in parallel
VERSIONS = f"torch {torch.__version__} (CUDA {torch.version.cuda}), timm {timm.__version__}"

os.environ["TZ"] = "Asia/Singapore"  # the Colab VM runs on UTC; use local time for batch names and log timestamps
time.tzset()
BATCH_NAME = datetime.now().strftime("%m%d_%H%M")  # this session's runs are named <BATCH_NAME>_<i>_<method>
BATCH_DIR = os.path.join(OUT_ROOT, "batches", BATCH_NAME)
os.makedirs(BATCH_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

LOG_FILE = None  # train_log.txt of the current run, set by run_experiment


def log(*args):
    """Print, and also append to this batch's batch_log.txt and the current run's train_log.txt (the report needs the training log)"""
    msg = " ".join(str(a) for a in args)
    print(msg, flush=True)
    stamp = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
    for path in (os.path.join(BATCH_DIR, "batch_log.txt"), LOG_FILE):
        if path:
            with open(path, "a") as f:
                f.write(stamp + msg + "\n")


ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
log(f"Batch {BATCH_NAME} -> {BATCH_DIR}")
log(f"Device {GPU_NAME} ({VRAM_GB} GB), bfloat16, {os.cpu_count()} CPUs, {ram_gb:.0f} GB RAM, "
    f"{shutil.disk_usage(DATA_DIR).free / 1e9:.0f} GB free disk")
log(f"Versions: {VERSIONS}")

## 3. Prepare data from Drive

- 数据只从 Drive 的 `DRIVE_DATA_DIR`（默认 `cs604/data/`）读取，要先用 `cs604_download_to_drive.ipynb` 存过去。找不到会直接报错。
- 每个新会话先把压缩包从 Drive 解压到 Colab 本地磁盘（`DATA_DIR`），之后从本地读图。直接从 Drive 一张张读 45 万张小图会非常慢。
- 图片保持原始大小（最长边 500 像素），不提前缩小：A100 运行时的 CPU 解码速度跟得上 GPU。
- 解压完会写一个标记文件，同一个会话里再运行就跳过这一步。
- 这里只用训练集。官方 val 集没有标签，提交文件只在主 notebook 里写。

In [ ]:
def prepare_split(split):
    """Extract {split}.tar.gz from Drive to the local Colab disk DATA_DIR/{split}/ and check the image count"""
    list_path = os.path.join(DRIVE_DATA_DIR, f"{split}.txt")
    tar_path = os.path.join(DRIVE_DATA_DIR, f"{split}.tar.gz")
    missing = [p for p in (list_path, tar_path) if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(f"Not found on Drive: {missing}. First run cs604_download_to_drive.ipynb to save the data to Drive")
    with open(list_path) as f:
        n_expected = sum(1 for line in f if line.strip())
    marker = os.path.join(DATA_DIR, f".{split}_ready")
    if not os.path.exists(marker):
        t0 = time.time()
        log(f"[{split}] extracting {tar_path} ...")
        subprocess.run(["tar", "-xzf", tar_path, "-C", DATA_DIR], check=True)
        n_found = sum(len(files) for _, _, files in os.walk(os.path.join(DATA_DIR, split)))
        assert n_found == n_expected, f"{split}: found {n_found} images, expected {n_expected}"
        open(marker, "w").close()
        log(f"[{split}] {n_found} images ready in {(time.time() - t0) / 60:.1f} min")
    return list_path


def read_list(path):
    """Each line is '<image path> <class id>', e.g. 'train/class_0000/9aef28bf-....jpg 0'"""
    items = []
    with open(path) as f:
        for line in f:
            parts = line.split()
            if parts:
                items.append((parts[0], int(parts[1])))
    return items


TRAIN_ALL = read_list(prepare_split("train"))
log(f"train {len(TRAIN_ALL):,} images | {NUM_WORKERS} loader workers")

## 4. Model (zero-shot / linear probe)

- `apply_method` 把所有参数（包括分类头）都设为不可训练，所以这个 notebook 没有训练循环。
- 分类头的权重完全来自 §6 的 `init_head`。

In [ ]:
def is_head(name):
    return name in ("head.weight", "head.bias")


def resolve_config(overrides):
    """Defaults from section 1 + one experiment's overrides → the full config of this run"""
    unknown = set(overrides) - set(RUN_KEYS)
    assert not unknown, f"unknown keys in experiment config: {sorted(unknown)}"
    cfg = SimpleNamespace(**{**BASE_CONFIG, **overrides})
    assert cfg.HEAD_INIT in ("proto", "lp"), f"HEAD_INIT must be 'proto' or 'lp', not {cfg.HEAD_INIT!r}"
    assert cfg.HOLDOUT_PER_CLASS > 0, "this notebook evaluates on the holdout set, so keep at least 1 image per class"
    if cfg.BATCH_SIZE is None:
        cfg.BATCH_SIZE = 256 if VRAM_GB >= 40 else 128 if VRAM_GB >= 24 else 64
    return cfg


def create_model(cfg):
    """Pretrained ViT-B/16 from timm; its 1000-class ImageNet head is replaced by a new, randomly initialized 10,000-class head"""
    return timm.create_model(cfg.MODEL_NAME, pretrained=True, num_classes=cfg.NUM_CLASSES).to(DEVICE)


def apply_method(net, cfg):
    """Zero-shot / linear probe: nothing is trained with gradients; the head was already set in section 6"""
    return net.requires_grad_(False)


def param_report(net, cfg, ctx):
    """Trainable parameter counts. The backbone is frozen; a linear-probe head was trained in section 6, so it counts as trainable"""
    n_train = sum(p.numel() for p in net.parameters() if p.requires_grad)  # 0: apply_method froze everything
    lp = cfg.HEAD_INIT == "lp"
    return {"backbone_params": ctx.n_backbone, "head_params": ctx.n_head,
            "head_mode": "linear probe" if lp else "class prototypes (not trained)",
            "trainable_excl_head": n_train, "trainable_pct": 100 * n_train / ctx.n_backbone,
            "trainable_incl_head": n_train + (ctx.n_head if lp else 0)}

## 5. Dataset and local holdout set

官方 val 集没有标签，所以从训练集每类留出 `HOLDOUT_PER_CLASS` 张图做本地 holdout，只用来测准确率，不参与训练。划分由 `SEED` 固定，每次实验都一样。这个 notebook 不训练主干，所以图片只做缩放和中心裁剪，没有数据增强。

In [ ]:
def split_holdout(items, k, seed):
    """Hold out k images of every class: shuffle each class's image list with a fixed seed and take the first k"""
    by_class = {}
    for path, label in items:
        by_class.setdefault(label, []).append(path)
    rng = random.Random(seed)
    train, holdout = [], []
    for label in sorted(by_class):
        paths = sorted(by_class[label])
        rng.shuffle(paths)
        holdout += [(p, label) for p in paths[:k]]
        train += [(p, label) for p in paths[k:]]
    return train, holdout


class ImageList(Dataset):
    def __init__(self, items, transform):
        self.items, self.transform = items, transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label = self.items[i]
        with Image.open(os.path.join(DATA_DIR, path)) as im:
            img = im.convert("RGB")
        return self.transform(img), label


BICUBIC = T.InterpolationMode.BICUBIC


def make_ctx(model, cfg):
    """Data setup for this run: train/holdout split, image transform, normalization values, param counts"""
    data_cfg = timm.data.resolve_model_data_config(model)
    size = data_cfg["input_size"][-1]
    train_items, holdout_items = split_holdout(TRAIN_ALL, cfg.HOLDOUT_PER_CLASS, cfg.SEED)
    return SimpleNamespace(
        img_size=size, train_items=train_items, holdout_items=holdout_items,
        mean=torch.tensor(data_cfg["mean"], device=DEVICE).view(1, 3, 1, 1) * 255,
        std=torch.tensor(data_cfg["std"], device=DEVICE).view(1, 3, 1, 1) * 255,
        # Only resize/crop on the CPU and output uint8; normalize on the GPU to save CPU work
        eval_tf=T.Compose([T.Resize(int(size / data_cfg["crop_pct"]), interpolation=BICUBIC),
                           T.CenterCrop(size), T.PILToTensor()]),
        n_backbone=sum(p.numel() for n, p in model.named_parameters() if not is_head(n)),
        n_head=sum(p.numel() for p in model.head.parameters()),
    )


class Loader(DataLoader):
    """Before starting new worker processes, run garbage collection in the main process. Otherwise an old, interrupted DataLoader is copied into the new workers,
    and when it is collected there it prints lots of harmless AssertionError: can only test a child process messages."""

    def __iter__(self):
        gc.collect()
        return super().__iter__()


def make_loader(items, ctx, cfg):
    """Center crops, in order, 2 × BATCH_SIZE images per batch (no gradients, so a big batch fits)"""
    return Loader(ImageList(items, ctx.eval_tf), batch_size=2 * cfg.BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)


def to_input(x, ctx):
    """uint8 images (B, 3, 224, 224) → normalized float32 on the GPU"""
    return (x.to(DEVICE, non_blocking=True).float() - ctx.mean) / ctx.std


def autocast():
    """bf16 mixed precision: matrix multiplies run in bfloat16, the weights stay float32"""
    return torch.autocast("cuda", dtype=torch.bfloat16)


def free_gpu():
    """Free the GPU memory left by the previous run and reset the peak memory stats"""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## 6. Classifier head

- `proto`（zero-shot）：用冻结的主干给所有训练图片提特征，把每一类特征的平均方向当作这一类的权重（类原型），再乘一个温度。不训练任何参数。
- `lp`（linear probe）：从类原型出发，在提前算好的特征上把分类头训练 `LP_EPOCHS` 个 epoch。只训练分类头，所以很快。

第二个实验复用第一个实验提好的特征（`FEATURE_CACHE`），但提特征的时间照样算进它的 GPU 时间，所以两个实验记的时间都是如实的。

In [ ]:
@torch.inference_mode()
def extract_features(net, items, ctx, cfg):
    """Features of the frozen backbone: the class-token output after the final LayerNorm, 768 numbers per image"""
    net.eval()
    feats, labels = [], []
    for x, y in tqdm(make_loader(items, ctx, cfg), desc="train features", leave=False):
        with autocast():
            f = net.forward_head(net.forward_features(to_input(x, ctx)), pre_logits=True)
        feats.append(f.half())
        labels.append(y.to(DEVICE))
    return torch.cat(feats), torch.cat(labels)


@torch.inference_mode()
def class_prototypes(feats, labels, num_classes):
    """Mean direction of each class: L2-normalize every feature, add them up per class, normalize the sums"""
    protos = torch.zeros(num_classes, feats.shape[1], device=feats.device)
    for i in range(0, len(feats), 65536):
        protos.index_add_(0, labels[i:i + 65536], F.normalize(feats[i:i + 65536].float(), dim=1))
    return F.normalize(protos, dim=1)


@torch.inference_mode()
def fit_logit_scale(protos, feats, labels, n=10000):
    """Pick a good temperature for cosine similarity, so the softmax of the prototype head is not too flat or too sharp"""
    idx = torch.randperm(len(feats), device=feats.device)[:n]
    cos = F.normalize(feats[idx].float(), dim=1) @ protos.T
    losses = {s: F.cross_entropy(cos * s, labels[idx]).item() for s in (5, 10, 15, 20, 30, 40, 60, 80)}
    return min(losses, key=losses.get)


def linear_probe(feats, labels, weight, bias, cfg):
    """Train only the 768 → 10,000 head on the cached features, starting from the prototype head"""
    head = nn.Linear(feats.shape[1], cfg.NUM_CLASSES).to(DEVICE)
    with torch.no_grad():
        head.weight.copy_(weight)
        head.bias.copy_(bias)
    opt = torch.optim.AdamW(head.parameters(), lr=cfg.LP_LR, weight_decay=cfg.LP_WD)
    total = cfg.LP_EPOCHS * math.ceil(len(feats) / cfg.LP_BATCH)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * s / total)))
    for _ in tqdm(range(cfg.LP_EPOCHS), desc="linear probe", leave=False):
        perm = torch.randperm(len(feats), device=feats.device)
        for i in range(0, len(feats), cfg.LP_BATCH):
            idx = perm[i:i + cfg.LP_BATCH]
            loss = F.cross_entropy(head(feats[idx].float()), labels[idx], label_smoothing=cfg.LABEL_SMOOTHING)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
    return head.weight.detach(), head.bias.detach()


FEATURE_CACHE = {}  # the features extracted by the first run, reused by the second run


def cached_features(model, ctx, cfg):
    """Frozen-backbone features of the training images. Extracted once; later runs with the same model and split reuse them"""
    key = (cfg.MODEL_NAME, cfg.HOLDOUT_PER_CLASS, cfg.SEED)
    if key in FEATURE_CACHE:
        log("Reusing the backbone features extracted by the previous run")
        return FEATURE_CACHE[key]
    t0 = time.time()
    feats, labels = extract_features(model, ctx.train_items, ctx, cfg)
    torch.cuda.synchronize()
    # Keep them on the CPU between runs, so they don't count toward the next run's GPU memory
    FEATURE_CACHE[key] = dict(feats=feats.cpu(), labels=labels.cpu(), seconds=time.time() - t0)
    log(f"Extracted features for {len(feats):,} train images in {FEATURE_CACHE[key]['seconds'] / 60:.1f} min")
    return FEATURE_CACHE[key]


def init_head(model, ctx, cfg):
    """Set the head from the class prototypes (zero-shot), or from a linear probe trained from them (lp).
    Returns the seconds spent; feature extraction counts too, even when the previous run already extracted the features"""
    cached = cached_features(model, ctx, cfg)
    t0 = time.time()
    feats, labels = cached["feats"].to(DEVICE), cached["labels"].to(DEVICE)
    protos = class_prototypes(feats, labels, cfg.NUM_CLASSES)
    scale = fit_logit_scale(protos, feats, labels)
    # The head works on unnormalized features: divide by the mean feature norm so logits ≈ scale × cosine similarity
    weight = protos * (scale / feats.float().norm(dim=1).mean())
    bias = torch.zeros(cfg.NUM_CLASSES, device=DEVICE)
    if cfg.HEAD_INIT == "lp":
        weight, bias = linear_probe(feats, labels, weight, bias, cfg)
    with torch.no_grad():
        model.head.weight.copy_(weight)
        model.head.bias.copy_(bias)
    torch.cuda.synchronize()
    seconds = cached["seconds"] + time.time() - t0
    source = "a linear probe started from the class prototypes" if cfg.HEAD_INIT == "lp" else "the class prototypes"
    log(f"Head set from {source} (prototype logit scale {scale}) in {seconds / 60:.1f} min incl. feature extraction, "
        f"counted in this run's GPU time")
    return seconds

## 7. Evaluation, inference latency and saving the head

- `evaluate` 在 holdout 图片上算 top-1 和 top-5。图片要重新过一遍主干，不用缓存的特征，相当于把整个流程从头验证一次。
- 推理延迟用随机输入测，只计模型的前向计算（不含读图和解码），用 bf16。batch 1 跑 100 次取平均；batch 256 跑 20 次，算吞吐量。
- `save_state` 只保存分类头，其他参数都没变。

In [ ]:
@torch.inference_mode()
def evaluate(net, loader, ctx):
    """Top-1 and top-5 accuracy on a labeled set"""
    net.eval()
    c1 = c5 = n = 0
    for x, y in loader:
        with autocast():
            top5 = net(to_input(x, ctx)).topk(5, dim=1).indices
        y = y.to(DEVICE)
        c1 += (top5[:, 0] == y).sum().item()
        c5 += (top5 == y[:, None]).any(dim=1).sum().item()
        n += len(y)
    return c1 / n, c5 / n


def save_state(net, path, **extra):
    """Save the classifier head, about 31 MB; nothing else has changed"""
    state = {n: p.detach().cpu() for n, p in net.named_parameters() if p.requires_grad or is_head(n)}
    torch.save({"state": state, **extra}, path)


@torch.inference_mode()
def measure_latency(net, batch_size, iters, ctx, warmup=10):
    """Average seconds per forward pass on random input: model compute only, no image reading or decoding"""
    net.eval()
    x = torch.randn(batch_size, 3, ctx.img_size, ctx.img_size, device=DEVICE)
    for _ in range(warmup):
        with autocast():
            net(x)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        with autocast():
            net(x)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters


def latency_report(net, ctx):
    lat_1 = measure_latency(net, 1, 100, ctx)
    lat_b = measure_latency(net, 256, 20, ctx)
    setting = f"{GPU_NAME}, bfloat16, no flip TTA"
    log(f"Inference latency (batch 1): {lat_1 * 1e3:.2f} ms/img | throughput (batch 256): {256 / lat_b:.0f} img/s | {setting}")
    return dict(latency_ms_bs1=lat_1 * 1e3, throughput_img_s=256 / lat_b, latency_setting=setting)

## 8. Full pipeline for one experiment

`run_experiment` 按顺序：建模型 → 设置分类头 → 冻结全部参数 → 保存分类头 → 在 holdout 上评估 → 测推理延迟 → 写 `summary.json`。结果在 Drive 的 `cs604/runs/<实验名>/`：`train_log.txt`、`config.json`、`summary.json` 和 `final.pt`。

In [ ]:
def run_experiment(overrides, run_name):
    """Run one experiment and return its summary. Everything is saved to cs604/runs/<run_name>/ on Drive"""
    global LOG_FILE
    cfg = resolve_config(overrides)
    run_dir = os.path.join(OUT_ROOT, "runs", run_name)
    os.makedirs(run_dir, exist_ok=True)
    LOG_FILE = os.path.join(run_dir, "train_log.txt")
    try:
        with open(os.path.join(run_dir, "config.json"), "w") as f:
            json.dump({**vars(cfg), "RUN_NAME": run_name, "overrides": overrides}, f, indent=2)
        log(f"\n========== {run_name} ==========")
        log(f"{METHOD}: head {cfg.HEAD_INIT}, holdout {cfg.HOLDOUT_PER_CLASS}/class"
            + (f", linear probe {cfg.LP_EPOCHS} epochs (lr {cfg.LP_LR}, wd {cfg.LP_WD}, batch {cfg.LP_BATCH})"
               if cfg.HEAD_INIT == "lp" else ""))
        log(f"Environment: {GPU_NAME} ({VRAM_GB} GB), bfloat16, {os.cpu_count()} CPUs | {VERSIONS}")
        random.seed(cfg.SEED)
        np.random.seed(cfg.SEED)
        torch.manual_seed(cfg.SEED)
        free_gpu()

        model = create_model(cfg)
        ctx = make_ctx(model, cfg)
        seconds = init_head(model, ctx, cfg)
        model = apply_method(model, cfg)
        params = param_report(model, cfg, ctx)
        log(f"Trainable (excl. head): {params['trainable_excl_head']:,} | head {ctx.n_head:,} params ({params['head_mode']})")
        save_state(model, os.path.join(run_dir, "final.pt"), config=vars(cfg))
        log(f"GPU time: {seconds / 3600:.2f} GPU-hours (feature extraction + classifier head; the backbone is not trained)")
        # The project doc asks for VRAM in the training log; reserved is closer to what nvidia-smi shows
        log(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB allocated, "
            f"{torch.cuda.max_memory_reserved() / 1e9:.1f} GB reserved by PyTorch (GPU has {VRAM_GB} GB)")

        top1, top5 = evaluate(model, make_loader(ctx.holdout_items, ctx, cfg), ctx)
        log(f"Final holdout: top-1 {top1:.2%}, top-5 {top5:.2%}")
        latency = latency_report(model, ctx)

        summary = dict(run=run_name, method=METHOD, head_init=cfg.HEAD_INIT, model=cfg.MODEL_NAME, overrides=overrides,
                       **params, holdout_top1=top1, holdout_top5=top5, **latency, gpu=GPU_NAME, vram_gb=VRAM_GB,
                       gpu_hours=seconds / 3600, peak_vram_gb=torch.cuda.max_memory_allocated() / 1e9,
                       peak_vram_reserved_gb=torch.cuda.max_memory_reserved() / 1e9, versions=VERSIONS)
        with open(os.path.join(run_dir, "summary.json"), "w") as f:
            json.dump(summary, f, indent=2)
        log(f"== {run_name} finished: holdout top-1 {top1:.2%} | {params['head_mode']} | {summary['gpu_hours']:.2f} GPU-h")
        return summary
    finally:
        LOG_FILE = None

## 9. Run the experiments

按顺序跑两个实验，实验名是 `<批次名>_<序号>_linear_<分类头>`。如果 zero-shot 的 holdout 准确率低于 25%（PDF 里约 43.7%），数据流程多半出了问题，直接停下。

In [ ]:
RESULTS = []
for i, exp in enumerate(EXPERIMENTS, 1):
    result = run_experiment(dict(exp), f"{BATCH_NAME}_{i}_{METHOD}_{exp.get('HEAD_INIT', HEAD_INIT)}")
    RESULTS.append(result)
    if result["head_init"] == "proto" and result["holdout_top1"] < 0.25:
        raise RuntimeError(f"Zero-shot holdout top-1 is only {result['holdout_top1']:.1%} (expected ~43.7%): "
                           "the data pipeline is probably broken, stopping. Please check the log.")

## 10. Results table

把这一批实验的结果整理成一张表，存到 Drive 的 `cs604/batches/<批次名>/results.md`（完整数据在同目录的 `results.json`），方便抄进报告和 slides。

In [ ]:
header = ("| Run | Classifier head | Holdout top-1 | Holdout top-5 | Trainable params (incl. head) | GPU-hours | "
          "Peak VRAM (GB) | Latency bs1 (ms) | Throughput (img/s) |\n|---|---|---|---|---|---|---|---|---|")
rows = [f"| {r['run']} | {r['head_mode']} | {r['holdout_top1']:.2%} | {r['holdout_top5']:.2%} | "
        f"{r['trainable_incl_head']:,} | {r['gpu_hours']:.2f} | {r['peak_vram_gb']:.1f} | "
        f"{r['latency_ms_bs1']:.2f} | {r['throughput_img_s']:.0f} |" for r in RESULTS]
table = header + "\n" + "\n".join(rows)
with open(os.path.join(BATCH_DIR, "results.md"), "w") as f:
    f.write(table + "\n")
with open(os.path.join(BATCH_DIR, "results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2)
log("\n" + table)

from IPython.display import Markdown, display
display(Markdown(table))

## 11. Auto disconnect (optional)

结果都已经存到 Drive 上了。`DISCONNECT_WHEN_DONE = True` 会在确认写完后释放运行时，停止消耗计算单元。同一个会话里还要跑别的实验就别打开。

In [ ]:
if DISCONNECT_WHEN_DONE:
    from google.colab import runtime
    log("All done; releasing the runtime ...")
    drive.flush_and_unmount()  # make sure all results are written to Drive
    runtime.unassign()